In [9]:
import os

import numpy as np
import matplotlib.pyplot as plt
from hamilton import driver
import torch
from tensordict import TensorDict

from world_machine_experiments import  shared
from world_machine_experiments.toy1d import base, Channels
from world_machine_experiments.toy1d.base.metrics import toy1d_metrics
from world_machine_experiments.shared.save_metrics import load_metrics

In [2]:
d = driver.Builder().with_modules(base, shared).build()


In [3]:
device = "cuda"

In [4]:
import torch
model = torch.load("toy1d_experiment0_protocol_test_pt/toy1d_experiment0_protocol_test/Base/run_0/toy1d_model.pt",
                   weights_only=False).to(device)

In [5]:
inputs = {"sequence_length": 1000,
                       "n_sequence": 10000,
                       "context_size": 200,
                       "batch_size": 32,
                       "learning_rate": 1e-3,
                       "cosine_annealing": True,
                       "cosine_annealing_T_mult": 1,
                       "cosine_annealing_T0": 25,
                       "weight_decay": 5e-5,
                       "accumulation_steps": 1,
                       "state_dimensions": [0],
                       "block_configuration": [Channels.MEASUREMENT, Channels.MEASUREMENT],
                       "device": device,
                       "state_control": "periodic",
                       "state_activation": "tanh",
                       "discover_state": True,
                       "sensory_train_losses": [Channels.MEASUREMENT],
                       "state_size": 128,
                       "positional_encoder_type": "alibi",
                       "n_attention_head": 4,

                       "seed":[0, 42]
            }

outputs = d.execute(["toy1d_data", "toy1d_dataloaders", "toy1d_criterion_set", "toy1d_model_untrained"], inputs=inputs)

In [6]:
toy1d_dataloaders = outputs["toy1d_dataloaders"]
toy1d_criterion_set = outputs["toy1d_criterion_set"]

In [7]:
metrics = toy1d_metrics(model, toy1d_dataloaders, toy1d_criterion_set)

Metrics Generation:   0%|          | 0/250 [00:00<?, ?it/s]c:\Users\eltsu\AppData\Local\Programs\Python\Python312\Lib\site-packages\pysdtw\sdtw_cuda.py:19: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\tensor\python_tensor.cpp:80.)
  gamma = torch.cuda.FloatTensor([gamma])
c:\Users\eltsu\AppData\Local\Programs\Python\Python312\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:748: NumbaPerformanceWarning: Grid size 96 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
c:\Users\eltsu\AppData\Local\Programs\Python\Python312\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:748: NumbaPerformanceWarning: Grid size 96 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(ms

In [8]:
metrics

{'normal': {'state_decoded_mse': 0.015315582975745201,
  'state_decoded_0.1sdtw': 0.11578676849603653,
  'measurement_mse': 0.009309467859566212,
  'measurement_0.1sdtw': 0.10765953361988068,
  'optimizer_loss': 0.01231252122670412},
 'use_state': {'state_decoded_mse': 0.2873659133911133,
  'state_decoded_0.1sdtw': 2.4636073112487793,
  'measurement_mse': 0.23954111337661743,
  'measurement_0.1sdtw': 4.034512519836426,
  'optimizer_loss': 0.26345348358154297},
 'prediction': {'state_decoded_mse': 0.7631391882896423,
  'state_decoded_0.1sdtw': 5.828394889831543,
  'measurement_mse': 0.42411959171295166,
  'measurement_0.1sdtw': 6.9378981590271,
  'optimizer_loss': 0.5936296582221985},
 'prediction_shallow': {'state_decoded_mse': 0.8546653389930725,
  'state_decoded_0.1sdtw': 6.4801740646362305,
  'measurement_mse': 0.4038183391094208,
  'measurement_0.1sdtw': 6.975329399108887,
  'optimizer_loss': 0.6292420625686646},
 'prediction_local': {'state_decoded_mse': 0.1136668249964714,
  'sta

In [13]:
metrics_origin = load_metrics("toy1d_experiment0_protocol_test\\Base\\run_0", "metrics")

In [15]:
metrics_origin["normal"]["optimizer_loss"] / metrics["normal"]["optimizer_loss"] 

0.20000640673186404

In [16]:
for name in metrics:
    print(name, metrics_origin[name]["optimizer_loss"] / metrics[name]["optimizer_loss"] )

normal 0.20000640673186404
use_state 0.2000425592350797
prediction 0.20012108094691547
prediction_shallow 0.19998162344545242
prediction_local 0.20001305277464695


In [21]:
(metrics_origin["normal"]["optimizer_loss"]*5)-metrics["normal"]["optimizer_loss"]

3.9441511034965515e-07

In [57]:
train_history = load_metrics("toy1d_experiment0_protocol_test\\Base\\run_0", "toy1d_train_history")

In [60]:
train_history["optimizer_loss_val"][-1]

np.float64(0.011893506161868572)

In [61]:
metrics_origin["normal"]["optimizer_loss"],  metrics["normal"]["optimizer_loss"]

(0.002462583128362894, 0.01231252122670412)

In [64]:
metrics["normal"]["optimizer_loss"]/train_history["optimizer_loss_val"][-1]

np.float64(1.0352305753352145)